In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/youngdaniel/bookgraph-comics/goodreads_reviews_comics_graphic.json
/kaggle/input/datasets/youngdaniel/bookgraph-comics/goodreads_interactions_comics_graphic.json
/kaggle/input/datasets/youngdaniel/bookgraph-comics/goodreads_books_comics_graphic.json
/kaggle/input/datasets/deepaktheanalyst/book-review/book_review.csv


In [2]:
# MARC — подготовка данных

from datasets import load_dataset
import pandas as pd, numpy as np

ds = load_dataset("hungnm/multilingual-amazon-review-sentiment-processed",
                  split="train")
df = ds.to_pandas()[["text", "stars", "language"]]
print(f"Загружено: {len(df):,}\n")

# --- распределение оценок по языкам: culture в чистом виде ---
print("=== РАСПРЕДЕЛЕНИЕ ЗВЁЗД ПО ЯЗЫКАМ (%) ===")
pivot = pd.crosstab(df["language"], df["stars"], normalize="index") * 100
print(pivot.round(1).to_string())

print("\n=== ДОЛЯ КРАЙНИХ ОЦЕНОК (1 и 5) ===")
extreme = (pivot[1] + pivot[5]).sort_values(ascending=False)
for lang, v in extreme.items():
    print(f"  {lang}: {v:.1f}%")

print("\n=== СРЕДНЯЯ ДЛИНА ОТЗЫВА ===")
df["len"] = df["text"].str.len()
for lang, v in df.groupby("language")["len"].mean().sort_values().items():
    print(f"  {lang}: {v:.0f} символов")

# --- сбалансированная выборка: поровну язык × оценка ---
PER_CELL = 2000        # 6 языков × 5 оценок × 2000 = 60 000
parts = []
for lang in sorted(df["language"].unique()):
    for star in [1, 2, 3, 4, 5]:
        cell = df[(df["language"] == lang) & (df["stars"] == star)]
        n = min(PER_CELL, len(cell))
        parts.append(cell.sample(n, random_state=42))
        if n < PER_CELL:
            print(f"  ⚠ {lang}/{star}★: только {n}")

marc = pd.concat(parts).sample(frac=1, random_state=42).reset_index(drop=True)
marc["label"] = marc["stars"] - 1        # 0..4 для обучения

print(f"\n=== ИТОГ ===")
print(f"Примеров: {len(marc):,} | языков: {marc['language'].nunique()}")
print(pd.crosstab(marc["language"], marc["stars"]).to_string())

marc[["text", "stars", "label", "language"]].to_parquet(
    "/kaggle/working/marc_balanced.parquet", index=False)
print("\n✓ /kaggle/working/marc_balanced.parquet")

README.md:   0%|          | 0.00/21.0 [00:00<?, ?B/s]

dataset_infos.json: 0.00B [00:00, ?B/s]

data/train-00000-of-00008-a8f46fd8ceb7b0(…):   0%|          | 0.00/41.2M [00:00<?, ?B/s]

data/train-00001-of-00008-450b7657a15d61(…):   0%|          | 0.00/41.2M [00:00<?, ?B/s]

data/train-00002-of-00008-8b2c9c4d5313f7(…):   0%|          | 0.00/41.1M [00:00<?, ?B/s]

...

data/test-00000-of-00001-6e49f9231189a55(…):   0%|          | 0.00/5.86M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1325529 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/47490 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/47488 [00:00<?, ? examples/s]

Загружено: 1,325,529

=== РАСПРЕДЕЛЕНИЕ ЗВЁЗД ПО ЯЗЫКАМ (%) ===
stars        1     2     4     5
language                        
de        25.0  24.8  25.0  25.2
en        25.2  24.7  24.8  25.3
es        25.0  25.0  25.0  25.0
fr        25.1  24.9  25.0  25.0
ja        25.0  25.0  24.9  25.0
zh        25.0  25.0  25.0  25.1

=== ДОЛЯ КРАЙНИХ ОЦЕНОК (1 и 5) ===
  en: 50.5%
  de: 50.2%
  fr: 50.1%
  zh: 50.1%
  es: 50.0%
  ja: 50.0%

=== СРЕДНЯЯ ДЛИНА ОТЗЫВА ===
  zh: 36 символов
  ja: 69 символов
  es: 95 символов
  fr: 99 символов
  en: 102 символов
  de: 122 символов
  ⚠ de/3★: только 0
  ⚠ en/3★: только 0
  ⚠ es/3★: только 0
  ⚠ fr/3★: только 0
  ⚠ ja/3★: только 0
  ⚠ zh/3★: только 0

=== ИТОГ ===
Примеров: 48,000 | языков: 6
stars        1     2     4     5
language                        
de        2000  2000  2000  2000
en        2000  2000  2000  2000
es        2000  2000  2000  2000
fr        2000  2000  2000  2000
ja        2000  2000  2000  2000
zh        2000  2000  2000  2